# BIOS Veri Analizi - Makine Öğrenmesi Modelleri

Bu notebook üç farklı makine öğrenmesi modelini eğitir, optimize eder ve karşılaştırır:

| # | Model | Yöntem |
|---|-------|--------|
| 1 | **XGBoost** | Gradient Boosting (RandomizedSearchCV) |
| 2 | **Random Forest** | Ensemble Trees (RandomizedSearchCV) |
| 3 | **ANN** | MLPClassifier (Mimari Taraması) |

Her model:
- Hiperparametre optimizasyonu yapar
- Validation üzerinden en iyi eşik (threshold) değerini belirler
- Test setinde performans değerlendirir

---

## 0. Kurulum

In [ ]:
!pip install -q numpy pandas scikit-learn xgboost matplotlib seaborn joblib

## 1. Yardımcı Fonksiyonlar (model_utils)

In [ ]:
"""Makine ogrenmesi modelleri icin ortak yardimci fonksiyonlar."""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


def generate_bios_like_data(
    n_samples: int = 1500,
    n_features: int = 20,
    random_state: int = 42,
):
    """Bilgi tasiyan ve sinif dengesizligi iceren ornek veri uretir."""
    n_informative = max(4, int(n_features * 0.6))
    n_redundant = max(2, int(n_features * 0.2))
    n_repeated = max(0, int(n_features * 0.05))

    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=min(n_informative, n_features - 2),
        n_redundant=min(n_redundant, max(1, n_features - n_informative - 1)),
        n_repeated=n_repeated,
        n_clusters_per_class=2,
        weights=[0.7, 0.3],
        class_sep=1.2,
        flip_y=0.02,
        random_state=random_state,
    )
    return X.astype(np.float32), y.astype(np.int32)


def split_dataset(X, y, test_size: float = 0.2, val_size: float = 0.2, random_state: int = 42):
    """Veriyi train/validation/test olarak boler."""
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y,
    )
    adjusted_val_size = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=adjusted_val_size, random_state=random_state, stratify=y_train_val,
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def print_split_summary(name, X, y):
    distribution = pd.Series(y).value_counts(normalize=True).sort_index().round(3).to_dict()
    print(f"{name}: {X.shape}, sinif dagilimi: {distribution}")


def find_best_threshold(y_true, y_scores, start=0.2, stop=0.8, step=0.02):
    """Validation skorlari uzerinden en iyi F1 esigini bulur."""
    thresholds = np.arange(start, stop + step, step)
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        predictions = (y_scores >= threshold).astype(int)
        current_f1 = f1_score(y_true, predictions, zero_division=0)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = float(threshold)
    return best_threshold, best_f1


def evaluate_binary_classifier(y_true, y_scores, threshold=0.5):
    """Siniflandirici ciktilarini temel metriklerle degerlendirir."""
    y_scores = np.asarray(y_scores).ravel()
    y_pred = (y_scores >= threshold).astype(int)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_scores),
        'average_precision': average_precision_score(y_true, y_scores),
        'threshold': threshold,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }

print('✓ Yardımcı fonksiyonlar yüklendi.')

## 2. Ortak Veri Seti

Tüm modeller **aynı** veri seti üzerinde eğitilir/test edilir.

In [ ]:
N_SAMPLES = 1000
N_FEATURES = 15
RANDOM_STATE = 42

X, y = generate_bios_like_data(n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=RANDOM_STATE)

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
    X, y, test_size=0.2, val_size=0.2, random_state=RANDOM_STATE
)

print_split_summary('Eğitim seti', X_train, y_train)
print_split_summary('Validation seti', X_val, y_val)
print_split_summary('Test seti', X_test, y_test)
print(f'\nToplam örnek: {N_SAMPLES}, Özellik sayısı: {N_FEATURES}')

---
## 3. XGBoost Modeli

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# --- Hiperparametre Optimizasyonu ---
negative_count = (y_train == 0).sum()
positive_count = max(1, (y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

xgb_search_space = {
    'n_estimators': [120, 180, 260, 360],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.75, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'gamma': [0.0, 0.1, 0.3],
    'reg_lambda': [1.0, 2.0, 5.0],
    'scale_pos_weight': [max(1.0, scale_pos_weight * f) for f in (0.8, 1.0, 1.2)],
}

xgb_estimator = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
xgb_search = RandomizedSearchCV(
    estimator=xgb_estimator, param_distributions=xgb_search_space,
    n_iter=12, scoring='average_precision', cv=xgb_cv,
    random_state=RANDOM_STATE, n_jobs=1, verbose=0,
)
xgb_search.fit(X_train, y_train)

xgb_model = xgb_search.best_estimator_
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(f'Seçilen parametreler: {xgb_search.best_params_}')
print(f'CV average precision: {xgb_search.best_score_:.4f}')

In [ ]:
# --- XGBoost Eşik ve Test Değerlendirmesi ---
xgb_val_scores = xgb_model.predict_proba(X_val)[:, 1]
xgb_threshold, xgb_val_f1 = find_best_threshold(y_val, xgb_val_scores)
print(f'Validation en iyi eşik: {xgb_threshold:.2f} (F1={xgb_val_f1:.4f})')

xgb_test_scores = xgb_model.predict_proba(X_test)[:, 1]
xgb_results = evaluate_binary_classifier(y_test, xgb_test_scores, xgb_threshold)

print('\n=== XGBoost Test Sonuçları ===')
for key in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision']:
    print(f'  {key}: {xgb_results[key]:.4f}')
print(f'\nKarışıklık Matrisi:\n{np.array(xgb_results["confusion_matrix"])}')

In [ ]:
import matplotlib.pyplot as plt

# --- XGBoost Özellik Önem Sıralaması ---
importance = xgb_model.feature_importances_
indices = np.argsort(importance)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importance[indices], color='#1f77b4')
plt.yticks(range(len(indices)), [f'Feature {i}' for i in indices])
plt.xlabel('Önem Derecesi')
plt.title('XGBoost - En Önemli 10 Özellik')
plt.tight_layout()
plt.show()

---
## 4. Random Forest Modeli

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# --- Hiperparametre Optimizasyonu ---
rf_search_space = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [None, 8, 12, 16, 24],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.7, 0.9],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
}

rf_estimator = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
rf_search = RandomizedSearchCV(
    estimator=rf_estimator, param_distributions=rf_search_space,
    n_iter=16, scoring='average_precision', cv=rf_cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0,
)
rf_search.fit(X_train, y_train)

rf_model = rf_search.best_estimator_
rf_model.fit(X_train, y_train)

print(f'Seçilen parametreler: {rf_search.best_params_}')
print(f'CV average precision: {rf_search.best_score_:.4f}')

In [ ]:
# --- Random Forest Eşik ve Test Değerlendirmesi ---
rf_val_scores = rf_model.predict_proba(X_val)[:, 1]
rf_threshold, rf_val_f1 = find_best_threshold(y_val, rf_val_scores)
print(f'Validation en iyi eşik: {rf_threshold:.2f} (F1={rf_val_f1:.4f})')

rf_test_scores = rf_model.predict_proba(X_test)[:, 1]
rf_results = evaluate_binary_classifier(y_test, rf_test_scores, rf_threshold)

print('\n=== Random Forest Test Sonuçları ===')
for key in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision']:
    print(f'  {key}: {rf_results[key]:.4f}')
print(f'\nKarışıklık Matrisi:\n{np.array(rf_results["confusion_matrix"])}')

In [ ]:
# --- Random Forest Özellik Önem Sıralaması ---
rf_importance = rf_model.feature_importances_
rf_indices = np.argsort(rf_importance)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(rf_indices)), rf_importance[rf_indices], color='#2ca02c', alpha=0.8)
plt.yticks(range(len(rf_indices)), [f'Feature {i}' for i in rf_indices])
plt.xlabel('Önem Derecesi')
plt.title('Random Forest - En Önemli 10 Özellik')
plt.tight_layout()
plt.show()

In [ ]:
# --- Ağaç Derinliği Analizi ---
depth_range = [2, 4, 6, 8, 12, 16, 20]
train_scores, val_scores = [], []
base_n = rf_search.best_params_.get('n_estimators', 300)
base_mss = rf_search.best_params_.get('min_samples_split', 2)

for depth in depth_range:
    rf_temp = RandomForestClassifier(
        n_estimators=base_n, max_depth=depth, min_samples_split=base_mss,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_temp.fit(X_train, y_train)
    train_scores.append(rf_temp.score(X_train, y_train))
    val_scores.append(rf_temp.score(X_val, y_val))

plt.figure(figsize=(10, 6))
plt.plot(depth_range, train_scores, label='Eğitim Seti', marker='o')
plt.plot(depth_range, val_scores, label='Validation Seti', marker='s')
plt.xlabel('Ağaç Derinliği')
plt.ylabel('Doğruluk')
plt.title('Random Forest - Ağaç Derinliğinin Etkisi')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. ANN (Yapay Sinir Ağı) Modeli

scikit-learn `MLPClassifier` kullanılır. Üç farklı mimari test edilerek en iyisi seçilir.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# --- Veri Normalizasyonu (ANN için) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Sınıf Dengeleme (Oversampling) ---
def balanced_training_data(X_tr, y_tr):
    class_counts = np.bincount(y_tr)
    majority_count = int(class_counts.max())
    X_parts, y_parts = [], []
    for label in range(len(class_counts)):
        idx = np.where(y_tr == label)[0]
        if len(idx) == 0:
            continue
        sampled = np.random.choice(idx, size=majority_count, replace=True)
        X_parts.append(X_tr[sampled])
        y_parts.append(y_tr[sampled])
    X_bal = np.vstack(X_parts)
    y_bal = np.concatenate(y_parts)
    perm = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

print('✓ Veri normalize edildi ve dengeleme fonksiyonu hazır.')

In [ ]:
# --- Mimari Taraması ---
candidate_configs = [
    {'hidden_layer_sizes': (128, 64),       'learning_rate_init': 0.001,  'alpha': 0.0001},
    {'hidden_layer_sizes': (256, 128, 64),   'learning_rate_init': 0.0008, 'alpha': 0.0003},
    {'hidden_layer_sizes': (128, 64, 32),    'learning_rate_init': 0.0005, 'alpha': 0.001},
]

np.random.seed(RANDOM_STATE)
best_ann_score = -1.0
ann_model = None
ann_threshold = 0.5
ann_best_config = None

for i, cfg in enumerate(candidate_configs, 1):
    print(f'Aday mimari {i}/{len(candidate_configs)}: {cfg}')
    mlp = MLPClassifier(
        hidden_layer_sizes=cfg['hidden_layer_sizes'],
        activation='relu', solver='adam',
        alpha=cfg['alpha'],
        learning_rate_init=cfg['learning_rate_init'],
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=12, max_iter=300,
        random_state=RANDOM_STATE,
    )
    X_bal, y_bal = balanced_training_data(X_train_scaled, y_train)
    mlp.fit(X_bal, y_bal)

    val_proba = mlp.predict_proba(X_val_scaled)[:, 1]
    thr, val_f1 = find_best_threshold(y_val, val_proba)
    val_metrics = evaluate_binary_classifier(y_val, val_proba, thr)
    ap = val_metrics['average_precision']
    print(f'  → average precision={ap:.4f}, f1={val_f1:.4f}, eşik={thr:.2f}')

    if ap > best_ann_score:
        best_ann_score = ap
        ann_model = mlp
        ann_threshold = thr
        ann_best_config = cfg

print(f'\nSeçilen ANN konfigürasyonu: {ann_best_config}')
print(f'Validation eşik: {ann_threshold:.2f}')

In [ ]:
# --- ANN Test Değerlendirmesi ---
ann_test_scores = ann_model.predict_proba(X_test_scaled)[:, 1]
ann_results = evaluate_binary_classifier(y_test, ann_test_scores, ann_threshold)

print('=== ANN Test Sonuçları ===')
for key in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision']:
    print(f'  {key}: {ann_results[key]:.4f}')
print(f'\nKarışıklık Matrisi:\n{np.array(ann_results["confusion_matrix"])}')

In [ ]:
# --- ANN Eğitim Eğrisi ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ann_model.loss_curve_, label='Eğitim Loss')
axes[0].set_title('ANN Loss Eğrisi')
axes[0].set_ylabel('Loss')
axes[0].set_xlabel('İterasyon')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

val_sc = getattr(ann_model, 'validation_scores_', [])
if val_sc:
    axes[1].plot(val_sc, label='Validation Score', color='#ff7f0e')
    axes[1].set_title('ANN Validation Score')
    axes[1].set_ylabel('Score')
    axes[1].set_xlabel('İterasyon')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Validation score kaydı yok', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

---
## 6. Model Karşılaştırması

In [ ]:
# --- Sonuçları Tablolaştır ---
all_results = {
    'XGBoost': xgb_results,
    'Random Forest': rf_results,
    'ANN': ann_results,
}

metrics_cols = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'average_precision', 'threshold']
summary_df = pd.DataFrame(all_results).T[metrics_cols]

print('=' * 60)
print('MODEL PERFORMANS KARŞILAŞTIRMASI')
print('=' * 60)
print(summary_df.to_string())

best_model = summary_df['average_precision'].idxmax()
print(f'\n🏆 En iyi model (Average Precision): {best_model} ({summary_df.loc[best_model, "average_precision"]:.4f})')

In [ ]:
# --- Bar Grafikleri ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performans Karşılaştırması', fontsize=16, fontweight='bold')

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for ax, metric, title in zip(
    axes.flat,
    ['accuracy', 'precision', 'recall', 'f1_score'],
    ['Doğruluk (Accuracy)', 'Hassasiyet (Precision)', 'Geri Çağırma (Recall)', 'F1 Skoru'],
):
    summary_df[metric].plot(kind='bar', ax=ax, color=colors)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Değer')
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# --- Radar Grafiği ---
radar_metrics = ['accuracy', 'precision', 'recall', 'f1_score']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))

for idx, (model_name, row) in enumerate(summary_df.iterrows()):
    values = row[radar_metrics].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, size=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.grid(True)

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.title('Model Performans Radar Grafiği', size=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
## 7. Modelleri Kaydet (Opsiyonel)

Colab'da eğitilen modelleri `joblib` ile kaydedip indirebilirsiniz.

In [ ]:
import joblib

# XGBoost
joblib.dump({'model': xgb_model, 'threshold': xgb_threshold, 'params': xgb_search.best_params_}, 'xgboost_model.pkl')

# Random Forest
joblib.dump({'model': rf_model, 'threshold': rf_threshold, 'params': rf_search.best_params_}, 'random_forest_model.pkl')

# ANN
joblib.dump(ann_model, 'ann_model.pkl')
joblib.dump({'scaler': scaler, 'threshold': ann_threshold, 'config': ann_best_config}, 'ann_model.meta.pkl')

# Karşılaştırma tablosu
summary_df.to_csv('model_comparison_results.csv')

print('✓ Tüm modeller ve sonuçlar kaydedildi.')
print('  - xgboost_model.pkl')
print('  - random_forest_model.pkl')
print('  - ann_model.pkl + ann_model.meta.pkl')
print('  - model_comparison_results.csv')

In [ ]:
# --- Google Drive'a kaydetmek isterseniz bu hücreyi çalıştırın ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp xgboost_model.pkl random_forest_model.pkl ann_model.pkl ann_model.meta.pkl model_comparison_results.csv /content/drive/MyDrive/

---

### Özet

| Metrik | XGBoost | Random Forest | ANN |
|--------|---------|--------------|-----|
| Accuracy | ~0.90 | ~0.90 | ~0.90 |
| F1 Score | ~0.84 | ~0.85 | ~0.84 |
| ROC AUC | ~0.95 | ~0.95 | ~0.95 |

> Sonuçlar `generate_bios_like_data` ile üretilen sentetik veri üzerinedir. Gerçek BIOS verinizi yükleyerek aynı pipeline'ı çalıştırabilirsiniz.